# 📊 Notebook 8: Evaluation — Đánh giá toàn diện

**⚠️ Chạy notebook 00 hoặc 01 trước để có data!**

In [ ]:
import subprocess, sys, os

try:
    import surprise
    import numpy as np
    import pandas as pd
    assert int(np.__version__.split('.')[0]) < 2, 'need numpy<2'
    assert int(pd.__version__.split('.')[0]) < 3, 'need pandas<3'
    print(f'✅ OK (numpy={np.__version__}, pandas={pd.__version__}, surprise={surprise.__version__})')
except Exception as e:
    print(f'📦 Installing... ({e})')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'numpy<2', 'pandas<3', 'scikit-surprise', 'scikit-learn',
        'matplotlib', 'seaborn', 'tqdm', '-q'])
    print('✅ Install xong! Runtime đang restart...')
    os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, json
from surprise import SVD, KNNWithMeans, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')
os.makedirs('results/charts', exist_ok=True)
os.makedirs('results/reports', exist_ok=True)

ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

print('✅ Evaluation ready!')

## 1. Train all models

In [ ]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

user_cf = KNNWithMeans(k=40, sim_option={'name':'cosine','user_based':True}, verbose=False)
user_cf.fit(trainset)

item_cf = KNNWithMeans(k=40, sim_option={'name':'cosine','user_based':False}, verbose=False)
item_cf.fit(trainset)

svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(trainset)

movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])
cosine_sim = cosine_similarity(tfidf_matrix)
movie_idx = pd.Series(movies.index, index=movies['movieId'])

print('✅ All models trained!')

## 2. RMSE & MAE

In [ ]:
preds = {'User-Based CF': user_cf.test(testset), 'Item-Based CF': item_cf.test(testset), 'SVD': svd.test(testset)}

results = []
for name, p in preds.items():
    r = accuracy.rmse(p, verbose=False)
    m = accuracy.mae(p, verbose=False)
    results.append({'Algorithm': name, 'RMSE': round(r, 4), 'MAE': round(m, 4)})
    print(f'{name}: RMSE={r:.4f}, MAE={m:.4f}')

df_results = pd.DataFrame(results).sort_values('RMSE')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['steelblue', 'coral', 'seagreen']
axes[0].bar(df_results['Algorithm'], df_results['RMSE'], color=colors, edgecolor='black')
axes[0].set_title('RMSE (↓ tốt)')
axes[1].bar(df_results['Algorithm'], df_results['MAE'], color=colors, edgecolor='black')
axes[1].set_title('MAE (↓ tốt)')
plt.tight_layout()
plt.savefig('results/charts/08_rmse_mae_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ranking Metrics

In [ ]:
def precision_at_k(actual, predicted, k):
    p = predicted[:k]
    return len(set(actual) & set(p)) / k if p else 0.0

def recall_at_k(actual, predicted, k):
    p = predicted[:k]
    return len(set(actual) & set(p)) / len(actual) if actual else 0.0

K = 10
sample_users = ratings['userId'].unique()[:500]

def eval_ranking(model, k=10):
    precs, recs = [], []
    for uid in sample_users:
        actual = ratings[(ratings['userId']==uid) & (ratings['rating']>=4)]['movieId'].values
        if len(actual) == 0: continue
        seen = set(ratings[ratings['userId']==uid]['movieId'])
        unseen = [m for m in ratings['movieId'].unique() if m not in seen]
        scores = {m: model.predict(uid, m).est for m in unseen}
        top_k = [m for m, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]]
        precs.append(precision_at_k(actual, top_k, k))
        recs.append(recall_at_k(actual, top_k, k))
    return {'P@10': round(np.mean(precs), 4), 'R@10': round(np.mean(recs), 4)}

print('Computing ranking metrics (~30s)...')
svd_r = eval_ranking(svd)
ucf_r = eval_ranking(user_cf)
icf_r = eval_ranking(item_cf)

for name, r in [('SVD', svd_r), ('User-CF', ucf_r), ('Item-CF', icf_r)]:
    print(f'  {name}: P@10={r["P@10"]}, R@10={r["R@10"]}')

## 4. Export

In [ ]:
all_results = {
    'User-Based CF': {'RMSE': results[0]['RMSE'], 'MAE': results[0]['MAE'], **ucf_r},
    'Item-Based CF': {'RMSE': results[1]['RMSE'], 'MAE': results[1]['MAE'], **icf_r},
    'SVD': {'RMSE': results[2]['RMSE'], 'MAE': results[2]['MAE'], **svd_r},
}

with open('results/reports/evaluation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('✅ Saved: results/reports/evaluation_results.json')
print(json.dumps(all_results, indent=2))

## Tổng kết

- SVD thường tốt nhất (RMSE thấp nhất)
- Hybrid kết hợp SVD + Content → cải thiện thêm
- Item-CF ổn định hơn User-CF